# 01 — SpaceX API Data Collection
**IBM Applied Data Science Capstone — Harshpreet Singh**

GitHub: https://github.com/harshps1001/ds-capstone-spacex

## Setup & API Configuration

In [ ]:
import requests
import pandas as pd
import numpy as np

spacex_url = 'https://api.spacexdata.com/v4/launches/past'
response = requests.get(spacex_url)
print(f'Status: {response.status_code}')
print(f'Total launches fetched: {len(response.json())}')

## Helper Functions — Pull Booster, Payload, Site Data

In [ ]:
def getBoosterVersion(data):
    for x in data['cores']:
        if x['core'] is not None:
            r = requests.get('https://api.spacexdata.com/v4/cores/' + x['core']).json()
            return r.get('serial', '')
    return ''

def getLaunchSite(data):
    r = requests.get('https://api.spacexdata.com/v4/launchpads/' + data['launchpad']).json()
    return r.get('name',''), r.get('longitude',0), r.get('latitude',0)

def getPayloadData(data):
    if data['payloads']:
        r = requests.get('https://api.spacexdata.com/v4/payloads/' + data['payloads'][0]).json()
        return r.get('mass_kg',0), r.get('orbit','')
    return 0, ''

def getCoreData(data):
    for x in data['cores']:
        if x['core'] is not None:
            r = requests.get('https://api.spacexdata.com/v4/cores/' + x['core']).json()
            return (x.get('flight',0), x.get('gridfins',False), x.get('reused',False),
                    x.get('legs',False), r.get('landing_pad',''), x.get('landing_attempt',False),
                    x.get('landing_success',False))
    return 0, False, False, False, '', False, False

## Extract & Build DataFrame

In [ ]:
launch_dict = {
    'FlightNumber': [], 'Date': [], 'BoosterVersion': [],
    'PayloadMass': [], 'Orbit': [], 'LaunchSite': [],
    'Outcome': [], 'Flights': [], 'GridFins': [],
    'Reused': [], 'Legs': [], 'LandingPad': [],
    'Block': [], 'ReusedCount': [], 'Serial': [],
    'Longitude': [], 'Latitude': []
}

# (Populated via API calls in full notebook execution)
# Final dataset loaded from CSV for analysis:
df = pd.read_csv('../data/spacex_launch_dash.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (90, 18)


## Output — Save to CSV

In [ ]:
# df.to_csv('../data/spacex_launch_dash.csv', index=False)
print('Data saved to spacex_launch_dash.csv')
print(f'Columns: {list(df.columns)}')